In [ ]:
!pip -q install tiktoken

In [ ]:
import torch
import tiktoken
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
d_model = 768
vocab_size = 50257  
max_seq_len = 1024

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

text = "Hello! How ore u? I cn't doing whatever райт нау емае он работает!!!"
token_ids = tokenizer.encode(text)

print(f"Размер словаря: {tokenizer.n_vocab}")
print(f"Количество токенов: {len(token_ids)}")
print(f"ID токенов: {token_ids}\n")

for t_id in token_ids:
    print(f"ID: {t_id:>6} -> {tokenizer.decode([t_id])!r}")

In [ ]:
class GPTModel(nn.Module):
    def __init__(self, vocab_size=50257, max_seq_len=1024, d_model=768):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        self.d_model = d_model
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape
        positions = torch.arange(seq_len, device=input_ids.device)
        return self.tok_emb(input_ids) + self.pos_emb(positions)

In [ ]:
embedding_model = GPTModel()
fake_input = torch.randint(0, vocab_size, (2, 8))  # 2 последовательности по 8 токенов
output = embedding_model(fake_input)
print("Форма выхода:", output.shape)  # torch.Size([2, 8, 768])

# Self-Attention — одна голова

| Элемент       | Что делает                 |
| ------------- | -------------------------- |
| **Q (Query)** | Что я ищу?                 |
| **K (Key)**   | Что у меня есть?           |
| **V (Value)** | Какую информацию передать? |
| **$QK^T$**    | Считаем сходство токенов   |
| **Softmax**   | Превращаем сходства в веса |
| **$AV$**      | Собираем новую информацию  |

### Формула

$$
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}
\left(
\frac{QK^T}{\sqrt{d_h}}
\right)V
$$


In [ ]:
class SingleHeadAttention(nn.Module):
    def __init__(self, d_model, head_size, max_seq_len):
        super().__init__()
        self.W_q = nn.Linear(d_model, head_size, bias=False)
        self.W_k = nn.Linear(d_model, head_size, bias=False)
        self.W_v = nn.Linear(d_model, head_size, bias=False)
        self.register_buffer("mask", torch.tril(torch.ones(max_seq_len, max_seq_len)))
        self.head_size = head_size

    def forward(self, x):
        _, seq_len, _ = x.shape
        q, k, v = self.W_q(x), self.W_k(x), self.W_v(x)
        scores = q @ k.transpose(-2, -1) * (self.head_size ** -0.5)
        scores = scores.masked_fill(self.mask[:seq_len, :seq_len] == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        return weights @ v

# Multi-Head Attention

| Элемент                 | Смысл                                        |
| ----------------------- | -------------------------------------------- |
| **Head**                | Отдельный способ искать связи между токенами |
| **$W_Q^h,W_K^h,W_V^h$** | Свои проекции для каждой головы              |
| **Concat**              | Объединяем результаты всех heads             |
| **$W_O$**               | Смешиваем полученную информацию              |

### Формула

$$
\operatorname{MHA}(X)
=
\operatorname{Concat}
\left(
\operatorname{head}_1,\ldots,\operatorname{head}_H
\right)W_O
$$

где

$$
\operatorname{head}_h
=
\operatorname{softmax}
\left(
\frac{Q_hK_h^T}{\sqrt{d_h}}
\right)V_h
$$


In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, num_heads, max_seq_len):
    super().__init__()
    self.d_model = d_model
    self.num_heads = num_heads
    self.head_size = d_model // num_heads
    self.c_attn = nn.Linear(d_model, 3 * d_model, bias=False)
    self.c_proj = nn.Linear(d_model, d_model, bias=False)
    self.register_buffer('mask', torch.tril(torch.ones(max_seq_len, max_seq_len)))

  def forward(self, x):

    B, T, C = x.shape
    q, k, v = self.c_attn(x).split(self.d_model, dim=2)

    # .view(B, T, self.num_heads, self.head_size) -> [B, T, num_heads, head_size]
    # .transpose(1, 2) меняет местами T и num_heads -> [B, num_heads, T, head_size]
    q = q.view(B, T, self.num_heads, self.head_size).transpose(1, 2)
    k = k.view(B, T, self.num_heads, self.head_size).transpose(1, 2)
    v = v.view(B, T, self.num_heads, self.head_size).transpose(1, 2)

    scores = (q @ k.transpose(-2, -1)) * (self.head_size ** -0.5) # [B, nh, T, T]
    M = scores.masked_fill(self.mask[:T, :T] == 0, float('-inf'))
    AttentionWeights = F.softmax(M, dim=-1)
    output_heads = AttentionWeights @ v # (B, nh, T, hs)
    concat_heads = output_heads.transpose(1, 2).contiguous().view(B, T, C) # (B, T, C)
    return self.c_proj(concat_heads)


## Вычисление внимания c PyTorch


In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, num_heads, max_seq_len):
    super().__init__()
    self.d_model = d_model
    self.num_heads = num_heads
    self.head_size = d_model // num_heads
    self.c_attn = nn.Linear(d_model, 3 * d_model, bias=False)
    self.c_proj = nn.Linear(d_model, d_model, bias=False)

  def forward(self, x):

    B, T, C = x.shape
    q, k, v = self.c_attn(x).split(self.d_model, dim=2)

    # .view(B, T, self.num_heads, self.head_size) -> [B, T, num_heads, head_size]
    # .transpose(1, 2) меняет местами T и num_heads -> [B, num_heads, T, head_size]
    q = q.view(B, T, self.num_heads, self.head_size).transpose(1, 2)
    k = k.view(B, T, self.num_heads, self.head_size).transpose(1, 2)
    v = v.view(B, T, self.num_heads, self.head_size).transpose(1, 2)

    output_heads = F.scaled_dot_product_attention(q, k, v, is_causal=True)
    concat_heads = output_heads.transpose(1, 2).contiguous().view(B, T, C) # (B, T, C)
    return self.c_proj(concat_heads)


In [ ]:
class FeedForward(nn.Module):
  def __init__(self, d_model):
    super().__init__()
    self.c_fc = nn.Linear(d_model, 4 * d_model)
    self.gelu = nn.GELU(approximate='tanh')
    self.c_proj = nn.Linear(4 * d_model, d_model)

  def forward(self, x):
    x = self.c_fc(x)
    x = self.gelu(x)
    x = self.c_proj(x)
    return x

In [ ]:
class GPTBlock(nn.Module):
  def __init__(self, d_model, num_heads, max_seq_len):
    super().__init__()
    self.ln_1 = nn.LayerNorm(d_model)
    self.attn = MultiHeadAttention(d_model, num_heads, max_seq_len)
    self.ln_2 = nn.LayerNorm(d_model)
    self.mlp = FeedForward(d_model)


  def forward(self, x):
    x = x + self.attn( self.ln_1(x) )
    x = x + self.mlp( self.ln_2(x) )
    return x

In [ ]:
class GPT2(nn.Module):
    def __init__(self, vocab_size=50257, max_seq_len=1024, d_model=768,
                 num_layers=12, num_heads=12):
        super().__init__()
        assert d_model % num_heads == 0
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        self.d_model = d_model
        self.num_layers = num_layers

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([
            GPTBlock(d_model, num_heads, max_seq_len) for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # Одни и те же веса для входных эмбеддингов и выходной проекции.
        self.lm_head.weight = self.tok_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape
        if seq_len > self.max_seq_len:
            raise ValueError(f"Слишком длинный вход: {seq_len} > {self.max_seq_len}")
        positions = torch.arange(seq_len, device=input_ids.device)
        x = self.tok_emb(input_ids) + self.pos_emb(positions)

        for block in self.blocks:
            x = block(x)

        return self.lm_head(self.ln_f(x))  # (batch, seq_len, vocab_size)

Для каждой последовательности `x = [t₁, …, tₙ]` ответом будет сдвинутая последовательность `y = [t₂, …, tₙ₊₁]`, на каждой позиции модель угадывает следующий токен. Добавляем причинную маску

In [ ]:
from pathlib import Path

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

data_path = Path("data.txt")

corpus = data_path.read_text(encoding="utf-8")

tokens = torch.tensor(tokenizer.encode(corpus), dtype=torch.long)
split = int(0.9 * len(tokens))
train_tokens, val_tokens = tokens[:split], tokens[split:]
print(f"Устройство: {device}; токенов: {len(tokens)} (обучение: {len(train_tokens)}, проверка: {len(val_tokens)})")

## Маленькая конфигурация и батчи

In [ ]:
config = dict(vocab_size=tokenizer.n_vocab, max_seq_len=32,
              d_model=64, num_layers=2, num_heads=4)
batch_size = 4
block_size = config["max_seq_len"]

if len(train_tokens) <= block_size or len(val_tokens) <= block_size:
    raise ValueError("Текст слишком короткий.")

def get_batch(data):
    starts = torch.randint(0, len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in starts])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in starts])
    return x.to(device), y.to(device)

model = GPT2(**config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
print(f"Параметров: {sum(p.numel() for p in model.parameters()):,}")

## Цикл обучения

In [ ]:
@torch.no_grad()
def estimate_loss(data, batches=3):
    model.eval()
    losses = []
    for _ in range(batches):
        x, y = get_batch(data)
        logits = model(x)
        loss = F.cross_entropy(logits.reshape(-1, model.vocab_size), y.reshape(-1))
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)

model.train()
steps = 60
for step in range(1, steps + 1):
    x, y = get_batch(train_tokens)
    logits = model(x)
    loss = F.cross_entropy(logits.reshape(-1, model.vocab_size), y.reshape(-1))

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step == 1 or step % 20 == 0:
        train_loss = estimate_loss(train_tokens)
        val_loss = estimate_loss(val_tokens)
        print(f"Шаг {step:>3}: train loss = {train_loss:.3f}, val loss = {val_loss:.3f}")

## Генерация и сохранение

Берём последний фрагмент длиной не больше контекста, предсказываем следующий токен и добавляем его ко входу (тут просто гриди стратегия: берем самый вероятный).

In [ ]:
@torch.no_grad()
def generate(prompt, max_new_tokens=40):
    model.eval()
    ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)
    if ids.size(1) == 0:
        raise ValueError("Введите непустой запрос")

    for _ in range(max_new_tokens):
        logits = model(ids[:, -block_size:])
        next_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        ids = torch.cat([ids, next_id], dim=1)
    return tokenizer.decode(ids[0].tolist())

print(generate("Отец мой"))

checkpoint_path = Path("gpt2_tiny.pt")
torch.save({"model_state_dict": model.state_dict(), "config": config,
            "tokenizer": "gpt2"}, checkpoint_path)
print(f"Веса сохранены в {checkpoint_path}")